***

Preparing Workspace

***

In [ ]:


import pandas as pd
from pathlib import Path
import plotly.express as px
from IPython.display import display
import sys
sys.path.append(str(Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen' / 'config'))
import plot as pt
import helpers

PATH_PLOTS = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
print('EXPORT Location: ' + str(PATH_PLOTS))
pd.options.display.float_format = '{:.2f}'.format


EXPORT=False


indicator = 'Cost_8'
plot_name = 'homelessness'
df_plot = pd.read_excel(PATH_PLOTS/'Data'/f"{indicator} Homelessness.xlsx", sheet_name='SACOG Region', skiprows=1)

df_plot = df_plot.dropna(subset = ['CoC Number'])
df_plot = df_plot[df_plot['CoC Number'] == 'SACOG Region Totals']
type = ['Overall', 'Sheltered', 'Unsheltered']
df_plot['Type'] = type
df_plot = df_plot.set_index('Type').reset_index()
df_plot = df_plot[['Type'] + helpers.sequence(2007, 2019, 2) + [2022, 2024]]
df_plot = df_plot[df_plot['Type'] != 'Overall']
df_plot = pd.melt(df_plot, id_vars = 'Type', var_name = 'Year', value_name = 'Count')
df_plot['Year'] = df_plot['Year'].astype(str)
display(df_plot.head())


fig = px.bar(df_plot, y='Count', x='Year', color='Type', barmode='group', color_discrete_map=pt.colors_homeless)
title = '<b>Estimated Homelessness Population</b>  <br><sup>Six-County Sacramento Region</sup> '
fig.update_yaxes(tick0=0, dtick=1000, range = [0, 10100], tickformat = ',.0f')
fig.update_traces(hovertemplate='%{y}')

# Add an annotation
fig.add_annotation(
    x=0.8, y=-0.08,
    # text="*No data collected in 2021<br>due to covid pandemic",
    text="*Data collection in 2021 delayed<br>due to the covid pandemic",
    showarrow=False,
    xanchor="left", 
    yanchor="top",
    font=dict(size=10),
    xref="paper", yref="paper"
)

pt.plot_agol(fig, EXPORT, title, indicator, plot_name)
